In [2]:
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index

URI = "neo4j://10.0.40.100:7687"
AUTH = ("neo4j", "sacf_sacf")

INDEX_NAME = "probill"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)
# Create Embedder object
# Note: An OPENAI_API_KEY environment variable is required here
embedder = OllamaEmbeddings(model="nomic-embed-text", host="http://10.0.40.49:11434")

# Initialize the retriever
retriever = VectorRetriever(driver, INDEX_NAME, embedder)

# Run the similarity search
query_text = "How do I do similarity search in Neo4j?"
response = retriever.search(query_text=query_text, top_k=5)


In [4]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import SseMcpToolAdapter, mcp_server_tools,StdioServerParams
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from pathlib import Path

mcp_server_path = str("/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mcp_tools/mcp-neo4j/servers/mcp-neo4j-cypher")
server_params = StdioServerParams(
    command="uv", args=[
        "--directory",
        mcp_server_path,
        "run",
        "mcp-neo4j-cypher",
        "--db-url",
        "bolt://10.0.40.49:7687",
        "--username", "neo4j",
        "--password",
        "sacf_sacf"
    ]
)

tools = await mcp_server_tools(server_params)

In [5]:
import json
tools_json = []
for tool in tools:
    # print(tool.name)
    tool_json = tool.dump_component().model_dump()
    tool_json["label"] = f"ne4j.{tool.name}"
    tool_json["config"]["name"] = f"ne4j.{tool.name}"
    tool_json["config"]["description"] = tool_json["description"]
    tools_json.append(tool_json)

print(json.dumps(tools_json, indent=2))

[
  {
    "provider": "autogen_ext.tools.mcp.StdioMcpToolAdapter",
    "component_type": "tool",
    "version": 1,
    "component_version": 1,
    "description": "Allows you to wrap an MCP tool running over STDIO and make it available to AutoGen.",
    "label": "ne4j.read-neo4j-cypher",
    "config": {
      "server_params": {
        "command": "uv",
        "args": [
          "--directory",
          "/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mcp_tools/mcp-neo4j/servers/mcp-neo4j-cypher",
          "run",
          "mcp-neo4j-cypher",
          "--db-url",
          "bolt://10.0.40.49:7687",
          "--username",
          "neo4j",
          "--password",
          "sacf_sacf"
        ],
        "encoding": "utf-8",
        "encoding_error_handler": "strict"
      },
      "tool": {
        "name": "read-neo4j-cypher",
        "description": "Execute a Cypher query on the neo4j database",
        "inputSchema": {
          "type": "object",
          "prop

In [1]:
import json
temp_json = {
  "nodes": [
    {
      "identifier": "Patient",
      "labels": ["Person"],
      "properties": {
        "age": 67,
        "sex": "Male"
      }
    },
    {
      "identifier": "Disease",
      "labels": ["LungCancer"],
      "properties": {
        "type": "NSCLC",
        "stage": "Stage IIIA"
      }
    },
    {
      "identifier": "Symptom",
      "labels": ["Symptom"],
      "properties": {
        "name": "Cough"
      }
    },
    {
      "identifier": "Biopsy",
      "labels": ["Biopsy"],
      "properties": {
        "result": "NSCLC Adenocarcinoma"
      }
    },
    {
      "identifier": "Mutation",
      "labels": ["Mutation"],
      "properties": {
        "type": "EGFR",
        "status": "Negative"
      }
    },
    {
      "identifier": "Treatment",
      "labels": ["Chemotherapy"],
      "properties": {
        "name": "Cisplatin + Pemetrexed"
      }
    },
    {
      "identifier": "DiagnosticTest",
      "labels": ["DiagnosticTest"],
      "properties": {
        "name": "CT Scan",
        "finding": "Right upper lobe mass"
      }
    }
  ],
  "relationships": [
    {
      "startNode": "Patient",
      "type": "HAS_CONDITION",
      "endNode": "Disease"
    },
    {
      "startNode": "Disease",
      "type": "PRESENTS_WITH",
      "endNode": "Symptom"
    },
    {
      "startNode": "Disease",
      "type": "DIAGNOSED_BY",
      "endNode": "Biopsy"
    },
    {
      "startNode": "Biopsy",
      "type": "HAS_MUTATION",
      "endNode": "Mutation"
    },
    {
      "startNode": "Disease",
      "type": "TREATED_WITH",
      "endNode": "Treatment"
    },
    {
      "startNode": "Patient",
      "type": "UNDERGOES",
      "endNode": "DiagnosticTest"
    }
  ]
}
print(temp_json)

{'nodes': [{'identifier': 'Patient', 'labels': ['Person'], 'properties': {'age': 67, 'sex': 'Male'}}, {'identifier': 'Disease', 'labels': ['LungCancer'], 'properties': {'type': 'NSCLC', 'stage': 'Stage IIIA'}}, {'identifier': 'Symptom', 'labels': ['Symptom'], 'properties': {'name': 'Cough'}}, {'identifier': 'Biopsy', 'labels': ['Biopsy'], 'properties': {'result': 'NSCLC Adenocarcinoma'}}, {'identifier': 'Mutation', 'labels': ['Mutation'], 'properties': {'type': 'EGFR', 'status': 'Negative'}}, {'identifier': 'Treatment', 'labels': ['Chemotherapy'], 'properties': {'name': 'Cisplatin + Pemetrexed'}}, {'identifier': 'DiagnosticTest', 'labels': ['DiagnosticTest'], 'properties': {'name': 'CT Scan', 'finding': 'Right upper lobe mass'}}], 'relationships': [{'startNode': 'Patient', 'type': 'HAS_CONDITION', 'endNode': 'Disease'}, {'startNode': 'Disease', 'type': 'PRESENTS_WITH', 'endNode': 'Symptom'}, {'startNode': 'Disease', 'type': 'DIAGNOSED_BY', 'endNode': 'Biopsy'}, {'startNode': 'Biopsy', 

In [21]:
import xml.etree.ElementTree as ET
import re
from typing import List, Dict, Set, Tuple

class DecisionRule:
    """
    Represents a single DMN rule with:
      - a list of (index, text) pairs for input conditions (the i-th condition belongs to i-th input)
      - a set of sanitized outcomes
      - a textual description
    """
    def __init__(self, rule_id: str, description: str):
        self.id = rule_id
        self.description = description
        # Store input conditions as (clauseIndex, rawText).
        # We'll link clauseIndex -> the i-th input label in the decisionTable.
        self.conditions: List[Tuple[int, str]] = []
        self.outcomes: Set[str] = set()

class DecisionNode:
    """
    Represents a DMN decision (with a single decisionTable) plus:
      - an ordered list of input labels (the i-th label -> i-th inputEntry in each rule)
      - an ordered list of output names (rarely needed but consistent with DMN)
      - a list of rules
    """
    def __init__(self, decision_id: str, name: str):
        self.id = decision_id
        self.name = name
        # We’ll store input labels in an *ordered* list to match them by index
        self.ordered_inputs: List[str] = []
        # Also keep a set of inputs for references (some DMN files have <dmn:input label="...">)
        self.inputs_set: Set[str] = set()

        # Output labels likewise
        self.ordered_outputs: List[str] = []
        self.outputs_set: Set[str] = set()

        # IDs of other decisions required
        self.requires: Set[str] = set()
        # The DMN rules
        self.rules: List[DecisionRule] = []

class DecisionGraph:
    def __init__(self):
        self.decisions: Dict[str, DecisionNode] = {}
        self.guideline_name = ""
        self.guideline_version = ""

    def sanitize(self, text: str) -> str:
        """Sanitize text for Cypher compatibility by removing invalid characters."""
        if text is None:
            return "Unknown"
        return re.sub(r'[^a-zA-Z0-9_]', '_', text)

    def parse_dmn(self, dmn_files: List[str]):
        """
        Parses DMN files, capturing the ordered inputs (the i-th <input>),
        the rules (<rule>), and the i-th <inputEntry> for each rule, etc.
        """
        NS = {'dmn': 'https://www.omg.org/spec/DMN/20191111/MODEL/'}

        for file_path in dmn_files:
            tree = ET.parse(file_path)
            root = tree.getroot()

            for decision_el in root.findall('dmn:decision', NS):
                decision_id = decision_el.get('id') or "UnnamedDecision"
                decision_name = decision_el.get('name') or "Unnamed Decision"

                # Create or reuse DecisionNode
                if decision_id not in self.decisions:
                    self.decisions[decision_id] = DecisionNode(decision_id, decision_name)
                node = self.decisions[decision_id]

                # Check for required decisions
                for req_decision in decision_el.findall('.//dmn:requiredDecision', NS):
                    href = req_decision.get('href') or ""
                    node.requires.add(href.replace('#', ''))

                # We assume there's a single <decisionTable> per <decision>
                dtable = decision_el.find('dmn:decisionTable', NS)
                if dtable is None:
                    # Some DMN might have literal expressions or other forms. Skip if no table?
                    continue

                # 1) Collect the <input> elements in order
                #    We'll store their labels in node.ordered_inputs
                inputs = dtable.findall('dmn:input', NS)
                node.ordered_inputs = []
                for inp_el in inputs:
                    label = inp_el.get('label') or ""
                    node.ordered_inputs.append(label)
                    node.inputs_set.add(label)

                # 2) Collect the <output> elements in order
                outputs = dtable.findall('dmn:output', NS)
                node.ordered_outputs = []
                for out_el in outputs:
                    label = out_el.get('label') or out_el.get('name') or ""
                    node.ordered_outputs.append(label)
                    node.outputs_set.add(label)

                # 3) For each <rule>, gather i-th <inputEntry> => the i-th input's condition
                for rule_el in dtable.findall('dmn:rule', NS):
                    rule_id = rule_el.get('id') or "UnnamedRule"
                    desc_el = rule_el.find('.//dmn:annotation', NS)
                    description_text = desc_el.text if desc_el is not None else ""
                    rule_obj = DecisionRule(rule_id, description_text)

                    # Collect the inputEntry in the same index as the <input>
                    input_entries = rule_el.findall('dmn:inputEntry', NS)
                    for idx, in_entry in enumerate(input_entries):
                        cond_text_el = in_entry.find('dmn:text', NS)
                        cond_text = cond_text_el.text if (cond_text_el is not None) else ""
                        rule_obj.conditions.append((idx, cond_text))

                    # Collect the outputEntry as outcomes
                    output_entries = rule_el.findall('dmn:outputEntry', NS)
                    for out_entry in output_entries:
                        text_el = out_entry.find('dmn:text', NS)
                        raw_out = text_el.text if (text_el is not None) else "Unknown"
                        rule_obj.outcomes.add(self.sanitize(raw_out))

                    node.rules.append(rule_obj)

    def parse_bpmn(self, bpmn_file: str):
        """
        Parse BPMN to get the guideline name/version.
        """
        namespace = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}
        tree = ET.parse(bpmn_file)
        root = tree.getroot()

        process = root.find(".//bpmn:process[@isExecutable='true']", namespace)
        if process is not None:
            guideline_name_version = process.get('name') or "UnknownGuideline"
            parts = guideline_name_version.rsplit(' ', 1)
            self.guideline_name = parts[0]
            if len(parts) > 1:
                self.guideline_version = parts[1]
            else:
                self.guideline_version = "1.0"
        else:
            self.guideline_name = "UnknownGuideline"
            self.guideline_version = "1.0"

    def generate_cypher(self) -> str:
        """
        Generate Cypher MERGE statements for:
          - A single Guideline node
          - One node per Decision
          - One node per Input / Output (unique variable names via counters)
          - One node per Rule
          - One node per Condition, linked to the correct Input via :CONDITION_ON
          - One node per Outcome
        """
        statements = []
        statements.append(
            f"MERGE (g:Guideline {{name: '{self.guideline_name}', version: '{self.guideline_version}'}})"
        )

        # Keep global counters for unique variable names
        decision_idx = 0
        input_idx = 0
        output_idx = 0
        rule_idx = 0
        cond_idx = 0
        outcome_idx = 0

        for decision_id, node in self.decisions.items():
            # Decision
            d_var = f"d_{decision_idx}"
            decision_idx += 1
            d_merge = f"MERGE ({d_var}:Decision {{name: '{node.name}'}})"
            statements.append(d_merge)
            statements.append(f"MERGE (g)-[:HAS_DECISION]->({d_var})")

            # Inputs
            # We'll keep track of the variable name for the i-th input
            input_vars = []
            for i, inp_label in enumerate(node.ordered_inputs):
                inp_var = f"inp_{input_idx}"
                input_idx += 1
                label_sanitized = self.sanitize(inp_label)
                statements.append(
                    f"MERGE ({inp_var}:Input {{name: '{label_sanitized}'}})"
                )
                statements.append(
                    f"MERGE ({d_var})-[:NEEDS_INPUT]->({inp_var})"
                )
                input_vars.append(inp_var)

            # Outputs
            output_vars = []
            for i, out_label in enumerate(node.ordered_outputs):
                out_var = f"out_{output_idx}"
                output_idx += 1
                out_sanitized = self.sanitize(out_label)
                statements.append(
                    f"MERGE ({out_var}:Output {{name: '{out_sanitized}'}})"
                )
                statements.append(
                    f"MERGE ({d_var})-[:HAS_OUTPUT]->({out_var})"
                )
                output_vars.append(out_var)

            # Merge each Rule
            for rule in node.rules:
                r_var = f"rule_{rule_idx}"
                rule_idx += 1
                desc_esc = (rule.description or "").replace("'", "\\'")
                statements.append(
                    f"MERGE ({r_var}:DecisionRule {{id: '{rule.id}', description: '{desc_esc}'}})"
                )
                statements.append(f"MERGE ({d_var})-[:HAS_RULE]->({r_var})")

                # For each condition, link to the correct input by index
                # rule.conditions is a list of (clauseIndex, rawText)
                for (clause_index, cond_text) in rule.conditions:
                    c_var = f"cond_{cond_idx}"
                    cond_idx += 1
                    cond_val = (cond_text or "Unknown").replace("'", "\\'")
                    # We'll store the name=some sanitized version, or 'Clause_{index}'
                    # plus the raw condition expression as 'value'
                    c_name = f"clause_{clause_index}"
                    c_name_sanitized = self.sanitize(c_name)

                    statements.append(
                        f"MERGE ({c_var}:Condition {{name: '{c_name_sanitized}', value: '{cond_val}'}})"
                    )
                    statements.append(
                        f"MERGE ({r_var})-[:HAS_CONDITION]->({c_var})"
                    )

                    # If we have an input for that clause_index, link Condition->Input
                    if 0 <= clause_index < len(input_vars):
                        target_inp_var = input_vars[clause_index]
                        statements.append(
                            f"MERGE ({c_var})-[:CONDITION_ON]->({target_inp_var})"
                        )

                # Merge outcomes
                for out_text in rule.outcomes:
                    outv = f"outcome_{outcome_idx}"
                    outcome_idx += 1
                    statements.append(
                        f"MERGE ({outv}:DecisionOutcome {{name: '{out_text}'}})"
                    )
                    statements.append(
                        f"MERGE ({r_var})-[:HAS_OUTCOME]->({outv})"
                    )

        return "\n".join(statements)
        
# Example usage
# if __name__ == "__main__":
dmn_files = ['./nccn/Confirmed diagnosis and clinical stage.dmn']
bpmn_file = './nccn/NCCN Breast Cancer 2023.2.bpmn'

graph = DecisionGraph()
graph.parse_dmn(dmn_files)
graph.parse_bpmn(bpmn_file)

cypher_script = graph.generate_cypher()
print(cypher_script)


MERGE (g:Guideline {name: 'NCCN Breast Cancer', version: '2023.2'})
MERGE (d_0:Decision {name: 'Confirmed diagnosis and clinical stage'})
MERGE (g)-[:HAS_DECISION]->(d_0)
MERGE (inp_0:Input {name: 'cT_stage'})
MERGE (d_0)-[:NEEDS_INPUT]->(inp_0)
MERGE (inp_1:Input {name: 'cN_stage'})
MERGE (d_0)-[:NEEDS_INPUT]->(inp_1)
MERGE (inp_2:Input {name: 'M_stage'})
MERGE (d_0)-[:NEEDS_INPUT]->(inp_2)
MERGE (inp_3:Input {name: 'histological_type'})
MERGE (d_0)-[:NEEDS_INPUT]->(inp_3)
MERGE (out_0:Output {name: 'go_to'})
MERGE (d_0)-[:HAS_OUTPUT]->(out_0)
MERGE (rule_0:DecisionRule {id: 'DecisionRule_192es8x', description: ''})
MERGE (d_0)-[:HAS_RULE]->(rule_0)
MERGE (cond_0:Condition {name: 'clause_0', value: '"cTis(DCIS)"'})
MERGE (rule_0)-[:HAS_CONDITION]->(cond_0)
MERGE (cond_0)-[:CONDITION_ON]->(inp_0)
MERGE (cond_1:Condition {name: 'clause_1', value: '"cN0"'})
MERGE (rule_0)-[:HAS_CONDITION]->(cond_1)
MERGE (cond_1)-[:CONDITION_ON]->(inp_1)
MERGE (cond_2:Condition {name: 'clause_2', value: 

In [34]:
import xml.etree.ElementTree as ET
import re
import json
from typing import List, Dict, Any

class OntologyBuilder:
    def __init__(self):
        self.decisions = {}

    def sanitize(self, text: str) -> str:
        """Sanitize text for node variable names by replacing invalid characters with underscores.
        If text is None, return 'Unknown'."""
        if text is None:
            return "Unknown"
        return re.sub(r'[^a-zA-Z0-9_]', '_', text)

    def is_unknown(self, text: str) -> bool:
        """Return True if text is None, empty, or equals 'Unknown' (case-insensitive)."""
        if text is None or text.strip() == "":
            return True
        return text.strip().lower() == "unknown"

    def parse_dmn(self, dmn_files: List[str]):
        NS = {'dmn': 'https://www.omg.org/spec/DMN/20191111/MODEL/'}

        for file_path in dmn_files:
            tree = ET.parse(file_path)
            root = tree.getroot()

            for decision_el in root.findall('dmn:decision', NS):
                decision_id = decision_el.get('id')
                decision_name = decision_el.get('name')

                decision = {
                    'id': decision_id,
                    'name': decision_name,
                    'inputs': [],
                    'outputs': [],
                    'rules': [],
                    'dependsOn': []
                }

                # Collect dependencies (required decisions)
                for req_decision in decision_el.findall('.//dmn:requiredDecision', NS):
                    href = req_decision.get('href')
                    if href:
                        decision['dependsOn'].append(href.replace('#', ''))

                dtable = decision_el.find('dmn:decisionTable', NS)
                if dtable is None:
                    continue

                # Collect inputs in order
                inputs = dtable.findall('dmn:input', NS)
                for inp_el in inputs:
                    label = inp_el.get('label')
                    decision['inputs'].append(label)

                # Collect outputs in order
                outputs = dtable.findall('dmn:output', NS)
                for out_el in outputs:
                    label = out_el.get('label') or out_el.get('name')
                    decision['outputs'].append(label)

                # Collect rules from the decision table
                for rule_el in dtable.findall('dmn:rule', NS):
                    rule = {'conditions': {}, 'outcomes': []}
                    input_entries = rule_el.findall('dmn:inputEntry', NS)
                    for idx, in_entry in enumerate(input_entries):
                        cond_text_el = in_entry.find('dmn:text', NS)
                        cond_text = (cond_text_el.text if (cond_text_el is not None and cond_text_el.text is not None)
                                     else "Unknown")
                        # Map the condition to the corresponding input label by index
                        if idx < len(decision['inputs']):
                            rule['conditions'][decision['inputs'][idx]] = cond_text
                        else:
                            rule['conditions'][f'input_{idx}'] = cond_text

                    output_entries = rule_el.findall('dmn:outputEntry', NS)
                    for out_entry in output_entries:
                        text_el = out_entry.find('dmn:text', NS)
                        outcome = (text_el.text if (text_el is not None and text_el.text is not None)
                                   else "Unknown")
                        rule['outcomes'].append(outcome)

                    decision['rules'].append(rule)

                self.decisions[decision_id] = decision

    def generate_json(self) -> str:
        return json.dumps(self.decisions, indent=2)

    def generate_cypher(self) -> str:
        cypher = []
        # Global set to keep track of declared ClinicalParameter nodes
        declared_params = set()
        
        # Process each Decision in the ontology
        for decision_id, decision in self.decisions.items():
            var_decision = self.sanitize(decision_id)
            cypher.append(f"MERGE ({var_decision}:Decision {{id:'{decision_id}', name:'{decision['name']}'}})")
            
            # Process inputs: ignore if unknown
            for input_name in decision['inputs']:
                if self.is_unknown(input_name):
                    continue
                var_input = self.sanitize(input_name)
                if var_input not in declared_params:
                    cypher.append(f"MERGE ({var_input}:ClinicalParameter {{name:'{input_name}'}})")
                    declared_params.add(var_input)
                cypher.append(f"MERGE ({var_decision})-[:HAS_INPUT]->({var_input})")
            
            # Link dependencies between decisions
            for dep in decision['dependsOn']:
                var_dep = self.sanitize(dep)
                cypher.append(f"MERGE ({var_dep}:Decision {{id:'{dep}'}})")
                cypher.append(f"MERGE ({var_decision})-[:DEPENDS_ON]->({var_dep})")
            
            # Process rules
            for idx, rule in enumerate(decision['rules']):
                rule_node = f"rule_{var_decision}_{idx}"
                cypher.append(f"MERGE ({rule_node}:DecisionRule {{id:'{rule_node}'}})")
                cypher.append(f"MERGE ({var_decision})-[:HAS_RULE]->({rule_node})")
                
                # Process conditions in the rule
                for param, cond in rule['conditions'].items():
                    if self.is_unknown(param):
                        continue
                    cond_val = cond if cond is not None else "Unknown"
                    cond_sanitized = self.sanitize(cond_val)
                    param_sanitized = self.sanitize(param)
                    condition_node = f"cond_{rule_node}_{param_sanitized}"
                    cypher.append(f"MERGE ({condition_node}:Condition {{param:'{param_sanitized}', value:'{cond_sanitized}'}})")
                    cypher.append(f"MERGE ({rule_node})-[:HAS_CONDITION]->({condition_node})")
                    
                    # Only create a ClinicalParameter node if not already declared\n
                    if param_sanitized not in declared_params:
                        cypher.append(f"MERGE ({param_sanitized}:ClinicalParameter {{name:'{param}'}})")
                        declared_params.add(param_sanitized)
                    cypher.append(f"MERGE ({condition_node})-[:CONDITION_ON]->({param_sanitized})")
                
                # Process outcomes in the rule; ignore unknown outcomes
                for outcome in rule['outcomes']:
                    if self.is_unknown(outcome):
                        continue
                    outcome_val = outcome
                    outcome_sanitized = self.sanitize(outcome_val)
                    outcome_node = f"outcome_{rule_node}_{outcome_sanitized}"
                    outcome_type = "TransitionOutcome" if re.match(r'^(BINV|DCIS|IBC)', outcome_val) else "ActionRecommendation"
                    cypher.append(f"MERGE ({outcome_node}:{outcome_type} {{name:'{outcome_val}'}})")
                    cypher.append(f"MERGE ({rule_node})-[:HAS_OUTCOME]->({outcome_node})")
                    if outcome_type == "TransitionOutcome":
                        var_target = self.sanitize(outcome_val)
                        cypher.append(f"MERGE ({var_target}:Decision {{id:'{outcome_val}'}})")
                        cypher.append(f"MERGE ({outcome_node})-[:LEADS_TO_DECISION]->({var_target})")
        
        return "\n".join(cypher)

# Usage Example
# if __name__ == '__main__':
#     builder = OntologyBuilder()
#     builder.parse_dmn([
#         '/mnt/data/Confirmed diagnosis and clinical stage.dmn',
#         '/mnt/data/Evaluation for metastatic therapy.dmn',
#         '/mnt/data/First-line therapy for HR-HER2+.dmn',
#         '/mnt/data/First-line therapy for HR+HER2-.dmn',
#         '/mnt/data/First-line therapy for HR+HER2+.dmn'
#     ])

#     # Output JSON structured representation
#     ontology_json = builder.generate_json()
#     with open('ontology_representation.json', 'w') as f:
#         f.write(ontology_json)

#     # Output Cypher script
#     cypher_script = builder.generate_cypher()
#     with open('ontology_graph.cypher', 'w') as f:
#         f.write(cypher_script)

#     print("JSON and Cypher scripts have been generated.")

# # Usage Example
# if __name__ == '__main__':
builder = OntologyBuilder()
builder.parse_dmn([
    './nccn/Confirmed diagnosis and clinical stage.dmn',
    './nccn/Evaluation for metastatic therapy.dmn',
    './nccn/First-line therapy for HR-HER2+.dmn',
    './nccn/First-line therapy for HR+HER2-.dmn',
    './nccn/First-line therapy for HR+HER2+.dmn'
])

# Output JSON structured representation
ontology_json = builder.generate_json()
with open('ontology_representation.json', 'w') as f:
    f.write(ontology_json)

# Output Cypher script
cypher_script = builder.generate_cypher()
with open('ontology_graph.cypher', 'w') as f:
    f.write(cypher_script)

print("JSON and Cypher scripts have been generated.")


JSON and Cypher scripts have been generated.
